### Data Loading and Preparation
In this initial block, we import the necessary data manipulation libraries and load the Iris dataset from `scikit-learn`. We extract the feature matrix into a pandas DataFrame `X` with appropriate column names, and assign the target labels to `y`.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris

raw = load_iris()
X = pd.DataFrame(raw.data, columns=raw.feature_names)
y = raw.target

### Support Vector Machine (SVM) Hyperparameter Exploration
Here, we evaluate an SVM classifier across different combinations of kernels (`linear`, `rbf`, `poly`) and K-Fold cross-validation splits (3 and 5). For each configuration, we construct a scikit-learn `Pipeline` that first normalizes the data using `StandardScaler` to ensure zero mean and unit variance, which is crucial for distance-based margins. We compute the macro F1-score to handle potential class imbalances and store the pipeline objects for subsequent decision boundary visualization.


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

svm_kernels = ["linear", "rbf", "poly"]
kfolds = [3, 5]

idx = 0
results = []
titles = []

for kernel in svm_kernels:
    for kfold in kfolds:
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(kernel=kernel))
        ])

        kf = KFold(n_splits=kfold, shuffle=True, random_state=42)
        scores = cross_val_score(pipeline, X, y, cv=kf, scoring="f1_macro")

        title = f"Kernel: {kernel} | K: {kfold}"
        titles.append(title)

        results.append({
            'kernel': kernel,
            'kf': kf,
            'pipeline': pipeline
        })

pd.DataFrame(results)

### SVM Decision Boundary Visualization
To visualize the classification margins, we reduce the feature space to just the first two dimensions (`X_2d`). We iterate through the previously saved SVM pipelines, fit them to this 2D subset, and plot their decision boundaries using `DecisionBoundaryDisplay`. The actual data points are overlaid as a scatter plot, allowing us to visually assess how different kernels and validation folds impact the model's spatial decision regions.


In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay
import matplotlib.pyplot as plt

X_2d = X.iloc[:, :-2]

fig, sub = plt.subplots(2,3, figsize=(12,8))
plt.subplots_adjust(wspace=0.3, hspace=0.4)
axes = sub.flatten()

for idx, result in enumerate(results):
    pipeline = result['pipeline']
    kernel = result['kernel']
    kfold = result['kf'].n_splits

    pipeline.fit(X_2d, y)
    ax = axes[idx]

    DecisionBoundaryDisplay.from_estimator(
        pipeline,
        X_2d,
        response_method="predict",
        cmap=plt.cm.coolwarm,
        alpha=0.8,
        ax=ax,
        xlabel=X_2d.columns[0],
        ylabel=X_2d.columns[1],
    )

    ax.scatter(X_2d.iloc[:, 0], X_2d.iloc[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolors="k")
    ax.set_title(f"Kernel: {kernel} | K: {kfold}")
    ax.set_xticks(())
    ax.set_yticks(())

plt.show()

### Multi-Layer Perceptron (MLP) Hyperparameter Grid Search
Next, we shift our focus to Neural Networks by evaluating an `MLPClassifier` with a single hidden layer of 100 neurons. We construct a parameter grid consisting of various activation functions, learning rates, and cross-validation folds. For each combination, we evaluate the pipeline (StandardScaler + MLP) using the macro F1-score. We also fit each model on the 2D feature subset to prepare for visualization, and finally display the grid search results sorted by the highest mean cross-validation score.


In [ ]:
from sklearn.neural_network import MLPClassifier


params = [
    (x,y,z)
    for x in ["relu","logistic"]
    for y in [0.001,0.01,0.03,0.003]
    for z in kfolds
]
results = []
models = []
for activation_function, learning_rate, k in params:
  clf = MLPClassifier(hidden_layer_sizes=(100,),
                      activation=activation_function,
                      solver='adam',
                      alpha=0.0001,
                      learning_rate='constant',
                      learning_rate_init=learning_rate,
                      max_iter=2000,
                      random_state=42,
                      early_stopping=False,
                      validation_fraction=0.1,
  )
  std = StandardScaler()
  pipeline = Pipeline([
    ('scaler', std),
    ('mlp', clf)
  ])
  kf = KFold(n_splits=k, shuffle=True, random_state=42)
  scores = cross_val_score(pipeline, X, y, cv=kf, scoring="f1_macro")
  results.append({
      'activation_function': activation_function,
      'learning_rate': learning_rate,
      'k': k,
      'mean_score': scores.mean()
  })
  clf.fit(X_2d,y)
  models.append(clf)

pd.DataFrame(results).sort_values(by="mean_score", ascending=False)

### MLP Decision Boundary Visualization
In this final block, we visualize the decision boundaries for the ensemble of trained MLP classifiers. We dynamically calculate the required grid layout based on the number of models evaluated. For each subplot, we render the model's 2D decision regions and overlay the ground-truth scatter points, annotating the axes with the specific activation function and initial learning rate used. Any empty subplots in the grid matrix are hidden for a cleaner presentation.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import DecisionBoundaryDisplay

n_models = len(models)
cols = 4
rows = (n_models + cols - 1) // cols

plt.subplots_adjust(wspace=0.3, hspace=0.4)
axes_flat = axes.flatten()

for idx, clf in enumerate(models):
    ax = axes_flat[idx]

    DecisionBoundaryDisplay.from_estimator(
        clf,
        X_2d,
        response_method="predict",
        cmap=plt.cm.coolwarm,
        alpha=0.8,
        ax=ax,
        xlabel=X_2d.columns[0],
        ylabel=X_2d.columns[1],
    )

    ax.scatter(X_2d.iloc[:, 0], X_2d.iloc[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolors="k")

    act = clf.activation
    lr = clf.learning_rate_init
    ax.set_title(f"{act} | LR: {lr}")
    ax.set_xticks(())
    ax.set_yticks(())

for idx in range(n_models, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.show()